# Dacon 이커머스 고객 분석 — 프로젝트 요약

2019년 온라인 커머스 거래를 바탕으로 ETL, EDA, 관측기간 코호트, RFM 등급,
등급별 프로파일과 실험 가설을 연결했다. 실제 신규 획득·캠페인 효과·인과 관계를
검증한 데이터는 아니므로 기술 분석과 실험 제안을 구분한다.


## 전체 분석 흐름

`원본 5개 테이블 → 검증된 거래라인 마스터 → EDA → 관측기간 코호트 → RFM 프로파일 → 등급별 실험 후보 → Tableau`

- 원본 52,924행을 모두 보존한다.
- 상품 매출은 할인 후 금액에 GST를 적용한 세후금액 합계(배송비 제외)로 `$4.84M`이다.
- 배송비는 거래당 한 번만 반영하며 총 `$220.5K`다.


## 파일별 핵심 역할

| 파일 | 역할 | 해석 범위 |
|---|---|---|
| 00 | ETL·조인·배송비/행 수 검증 | 분석 데이터 생성 |
| 01 | 매출·고객·상품라인·마케팅 기술통계 | 인과 효과 아님 |
| 02 | 2019년 관측기간 첫 구매 코호트 | 실제 신규 코호트 아님 |
| 03 | RFM 운영 점수·등급 프로파일 | 미래 성과 검증 필요 |
| 04~07 | 등급별 프로파일과 실험 후보 | 등급 정의의 영향 포함 |


## RFM 등급 설계

R/F/M 구간과 PC1 로딩값의 절댓값을 정규화한 가중치(설명 분산 69.4%)로 20~100점
점수를 만들고 설명 가능한 고정 컷을 적용했다. PCA 로딩은 비즈니스 중요도나 예측 중요도가 아니다. 새 데이터에서 운영할
때는 구간·가중치·버전을 함께 고정하고 과거 등급이 미래 매출·복귀를 예측하는지 검증한다.

| 등급 | 고객수 | 고객 비중 | 1인당 상품매출 | 연간 재방문율 |
|---|---:|---:|---:|---:|
| Diamond | 58 | 4.0% | `$17,482` | 94.8% |
| Platinum | 154 | 10.5% | `$8,677` | 88.3% |
| Gold | 243 | 16.6% | `$4,848` | 74.1% |
| Silver | 315 | 21.5% | `$2,484` | 56.8% |
| Bronze | 698 | 47.5% | `$758` | 26.4% |


## 핵심 관찰

1. Nest-USA와 상위 등급에 상품 매출이 집중된다.
2. 관측기간 첫 구매 후 +1개월 코호트 리텐션은 고객 가중 9.5%로 낮지만 이를 실제 신규 온보딩 성과로 해석할 수 없다.
3. 등급별 매출·재방문 차이는 RFM 설계의 영향을 포함하므로 등급 프로파일로 본다.
4. Silver·Gold·Platinum에는 누적 구매액이 높지만 최근 구매가 줄어든 고객이 존재한다.
5. 고정 주기·계절성·쿠폰 효과는 확정되지 않았으며 실험 또는 다년도 데이터가 필요하다.


## 실험 우선순위

| 우선순위 | 대상 | 실험 | 핵심 지표 |
|---|---|---|---|
| 1 | Bronze 최근·저빈도 220명<br>(관측기간 첫 구매 조건) | 2개 그룹 50:50, 그룹당 110명<br>30일 vs 60일 메시지 | 후속 30일 내 다른 날 재구매율 |
| 2 | Silver 이탈위험 55명 + Gold 이탈위험 44명 = 99명<br>(장기 미방문) | 3개 그룹 1:1:1·등급 층화, 그룹당 33명<br>60일 / 90일 / 120일 접촉 시점 | 후속 30일 복귀율·마진·수신거부율 |
| 3 | Platinum 미방문 33명<br>(Recency > 90일) | 2개 그룹 50:50, 17명 / 16명<br>일반 메시지 vs 개인화 메시지 | 후속 30일 복귀율·할인비용률 |
| 4 | Diamond·Platinum 212명<br>(Diamond 58 + Platinum 154) | 2개 그룹 50:50, 그룹당 106명<br>쿠폰 vs 등급 혜택 | 증분 재구매율·객단가 |

Platinum 미방문 33명은 표본이 작아 단일 실험보다 관찰을 우선한다.

실험 전 고객 단위 랜덤 배정, MDE·표본 수 계산, 중복 노출 방지와 가드레일을 확정한다.


## 한계와 다음 단계

- 2019년 1개년 합성 데이터라 반복 계절성을 검증할 수 없다.
- 마진 데이터가 없어 ROI는 산출할 수 없다. 광고비가 있는 범위에서는 매출/광고비 배수로만 표현했다.
- 90일 장기 미방문은 시나리오 기준이며 60/90/120/150일 민감도와 운영 비용을 함께 본다.
- 다음 단계는 과거 기간 등급 → 미래 기간 매출·복귀의 out-of-time 검증이다.
- 등급 산출 기준을 버전으로 고정해야 한다. 데이터 정제 과정에서 PCA 가중치가 소수점 넷째 자리 수준으로 변했을 때 등급 경계 고객이 이동해 세그먼트 인원과 파생 지표가 함께 바뀌었다.
- 같은 이름의 지표가 집계 단위에 따라 다른 값을 갖는다. 쿠폰 사용률은 상품 라인 33.8%, 거래 47.6%, 고객 93.6%로 갈리므로 인용 시 단위를 병기한다.
- 세그먼트 규모가 33~295명으로 작아 개별 실험의 검정력은 제한적이다. 실행 시 적격 모집단 확장이나 기간 누적 배정이 필요하다.


## 대시보드 연결

Tableau는 등급별 고객수·상품 매출·연간 재방문율·세그먼트·관측기간 코호트를 보여준다.
등급으로 정의된 지표 차이는 독립 성과 검증이 아니라 프로파일이며, 월별 전체 매출은
등급 필터와 분리된 전사 추세다.

등급별 고객 수·매출 기여·재방문율과 실험 우선순위는 Tableau 대시보드로 구성했다. https://public.tableau.com/app/profile/.16528220/viz/_17872369187540/01
